# Code was run on Colab Pro

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import collections
import torch.optim as optim
from torch.optim import Optimizer
import time
import matplotlib.pyplot as plt

from AdamW          import AdamW
from utils          import utility, misreportUtility, misreportOptimization, trueUtility, loss
from networks       import AdditiveMechanism, Misreports,AllocationNet,PaymentNet
from restrictedAdam import Adam 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.cuda.set_device(3)

# Set Random Seed 

In [3]:
# Initializing seeds
torch.manual_seed(3)
np.random.seed(3)

# Testing Function

In [4]:
def test(nBatch, nbrInit, R, gamma=0.001, minimum=0, maximum=1):
    
    """ This function computes the regret and payment of mechanism on a test set of size nBatch
        The optimal misreport is computed by optimizing the utility function (not by using the Misreport network)
        for R gradient steps (of stepsize gamma) and starting from nbrInit initialization, we only keep the best misreport
        To compute the regret we evaluate the mechanism at the misreport and compare to the valuation
        minimum and maximum indicate the range of the valuations
    """
    
    true = np.random.rand(nBatch,nAgent,nObject)

    localMisreports     = np.random.rand(nBatch,nbrInit,nAgent,nObject)
    batchMisreports     = torch.tensor(localMisreports).float().to(device)
    batchTrueValuations = torch.tensor(true).float().to(device)
    batchMisreports.requires_grad = True
    
    opt = Adam([batchMisreports], lr=gamma)
    
    for k in range(R):
        advU         = misreportUtility(mechanism,batchTrueValuations,batchMisreports)
        los          =  -1*torch.mean(advU).to(device)
        los.backward()
        opt.step(restricted= True, min=minimum, max=maximum)
        opt.zero_grad()
    
    misReportUtilityMax  = torch.max(advU, dim =1)[0]
    mechanism.zero_grad()
    allocation, payment = mechanism(batchTrueValuations)
    regret = F.relu(misReportUtilityMax -utility(batchTrueValuations, allocation, payment))
    mregret= torch.sum(torch.mean(regret, dim=0)).to(device)
    mregret= float(mregret.cpu().detach().numpy())

    with torch.no_grad():
        l,rMean,p = loss(payment, regret)

    testRegret.append(mregret)
    testPayment.append(float(p.detach().cpu().numpy() ))
    testOptimal.append(float((-l).detach().cpu().numpy())**2)
    print("Total regret: ",'{0:.5f}'.format(mregret), "Average regret per bidder: ",'{0:.5f}'.format(mregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float((-l).detach().cpu().numpy())**2), " payment: ",'{0:.3f}'.format(float(p.detach().cpu().numpy() )))

# Initializing Networks

In [5]:
nAgent   = 5
nObject  = 10

# Parameters for the mechanism (payment and allocation network)
nLayersAllocation   = 8
nLayersPayment      = 8
widthAllocation     = 200
widthPayment        = 200

# Parameters for the misreport network
nLayersMisreport    = 8
widthMisreport      = 200

gamma              = 0.001 
testBatch          = 10000

nExperiments       = 200000
batchSize          = 500
nbrBatches         = int(nExperiments/batchSize)



alloc_net = AllocationNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_alloc = AdamW(alloc_net.parameters(), lr=1e-3)

pay_net   = PaymentNet(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
opt_pay   = AdamW(pay_net.parameters(),   lr=1e-3)

mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.001)

misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

In [6]:
testRegret    = []
testMaxRegret = []
testPayment   = []
testOptimal   = []
testTime      = []
testIteration = [0]

# range of valuations
minimum            = 0
maximum            = 1

In [7]:
@torch.no_grad()
def myerson_itemwise_allocation_payment(values, reserve=0.5):
    """
    values:  (B, A, O)  估值矩阵
    reserve: 保留价
    return:  alloc (B, A, O), pay_myr (B, A)
    """
    B, A, O = values.shape
    top2 = values.topk(k=2, dim=1)
    v1, idx1 = top2.values[:, 0, :], top2.indices[:, 0, :]  
    v2 = top2.values[:, 1, :]                              

    win = (v1 >= reserve).float()        # (B,O)
    price = torch.maximum(v2, torch.full_like(v2, reserve)) * win

    alloc = torch.zeros(B, A, O, device=values.device)
    alloc.scatter_(1, idx1.unsqueeze(1), win.unsqueeze(1))

    pay_myr = alloc * price.unsqueeze(1)  # (B,A)
    return alloc, pay_myr

# Training

In [8]:
R=10000
reserve=0.5
print("Train AllocationNet with Myerson supervision")
for t in range(1,60*nbrBatches+1):
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    alloc_pred = alloc_net(values)
    loss_alloc = F.mse_loss(alloc_pred, alloc_myr)

    opt_alloc.zero_grad()
    loss_alloc.backward()
    opt_alloc.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_alloc.item():.6f}")

Train AllocationNet with Myerson supervision
loss=0.015182
loss=0.013497
loss=0.012883
loss=0.012823
loss=0.011369
loss=0.011027
loss=0.010868
loss=0.010682
loss=0.009563
loss=0.008357
loss=0.008197
loss=0.008100
loss=0.007348
loss=0.008399
loss=0.007958
loss=0.007606
loss=0.007812
loss=0.007088
loss=0.006947
loss=0.006718
loss=0.006443
loss=0.006439
loss=0.006767
loss=0.006242
loss=0.006399
loss=0.005883
loss=0.005684
loss=0.005502
loss=0.005081
loss=0.005848


In [9]:
print("Train PaymentNet with Myerson supervision")

for t in range(1,60*nbrBatches+1):
    values = torch.rand(batchSize, nAgent, nObject, device=device)

    with torch.no_grad():
        alloc_myr, pay_myr = myerson_itemwise_allocation_payment(values, reserve=reserve)

    payments_pred = pay_net(values, alloc_myr)

    loss_pay = F.mse_loss(payments_pred, pay_myr)

    opt_pay.zero_grad()
    loss_pay.backward()
    opt_pay.step()
    if t % (2*nbrBatches)==0 :
        print(f"loss={loss_pay.item():.6f}")

Train PaymentNet with Myerson supervision
loss=0.001327
loss=0.000962
loss=0.000552
loss=0.000401
loss=0.000301
loss=0.000231
loss=0.000222
loss=0.000189
loss=0.000172
loss=0.000190
loss=0.000138
loss=0.000121
loss=0.000116
loss=0.000126
loss=0.000105
loss=0.000093
loss=0.000098
loss=0.000082
loss=0.000085
loss=0.000071
loss=0.000077
loss=0.000066
loss=0.000067
loss=0.000059
loss=0.000070
loss=0.000060
loss=0.000053
loss=0.000052
loss=0.000054
loss=0.000049


In [10]:
duration   = 0
R          = 100

i=0
mechanism            = AdditiveMechanism(nAgent, nObject, nLayersAllocation, widthAllocation).to(device)

mechanism.alloc_net.load_state_dict(alloc_net.state_dict())
mechanism.payment_net.load_state_dict(pay_net.state_dict())
optimizerMechanism   = AdamW(mechanism.parameters(), lr=0.001)

print("Initial Test")
test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

for t in range(1,60*nbrBatches+1):
    
    # Reinitialize Misreport network periodically at the beginning of training
    if (t%(2*nbrBatches) ==1):
      if   t< 20*nbrBatches+2 :
    
        misreport            = Misreports(nAgent,nObject,nLayersMisreport, widthMisreport).to(device)
        optimizerMisreport   = AdamW(misreport.parameters(), lr=0.001)

    batchTrueValuations = torch.tensor(np.random.rand(batchSize,nAgent,nObject)).float().to(device)
    
    # Optimize Misreport Network for R steps
    for k in range(R):
  
        misreports          = misreport(batchTrueValuations).unsqueeze(1)
        mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)
        mLoss               = torch.sum(torch.mean(-mUtility,dim=0))

        optimizerMisreport.zero_grad()
        mLoss.backward()
        optimizerMisreport.step()

    
    # Optimize Mechanism network for one step
    misreports          = misreport(batchTrueValuations).unsqueeze(1)
    mUtility            = misreportUtility(mechanism,batchTrueValuations,misreports).squeeze(1)

    allocation, payment = mechanism(batchTrueValuations)

    regret     = F.relu(mUtility -utility(batchTrueValuations, allocation, payment))
    l,rMean,p = loss(payment, regret)
        
    optimizerMechanism.zero_grad()

    l.backward()

    optimizerMechanism.step()
    
    # Test mechanism periodically
    if t % (2*nbrBatches)==0 :
        print("Batch: ", 2*int(t/(2*nbrBatches)))
        testTime.append(duration)
        testIteration.append(t/nbrBatches)
        test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Initial Test


/home/wkw/ysy/women (1)/restrictedAdam.py:103: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ../torch/csrc/utils/python_arg_parser.cpp:1050.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)


Total regret:  0.28704 Average regret per bidder:  0.05741  Optimal Revenue:  3.153  payment:  6.752
Batch:  2
Total regret:  0.02767 Average regret per bidder:  0.00553  Optimal Revenue:  5.575  payment:  6.529
Batch:  4
Total regret:  0.02623 Average regret per bidder:  0.00525  Optimal Revenue:  5.847  payment:  6.793
Batch:  6
Total regret:  0.03135 Average regret per bidder:  0.00627  Optimal Revenue:  5.727  payment:  6.768
Batch:  8
Total regret:  0.02248 Average regret per bidder:  0.00450  Optimal Revenue:  5.868  payment:  6.733
Batch:  10
Total regret:  0.02493 Average regret per bidder:  0.00499  Optimal Revenue:  5.914  payment:  6.836
Batch:  12
Total regret:  0.02393 Average regret per bidder:  0.00479  Optimal Revenue:  5.910  payment:  6.810
Batch:  14
Total regret:  0.02467 Average regret per bidder:  0.00493  Optimal Revenue:  5.945  payment:  6.864
Batch:  16
Total regret:  0.02006 Average regret per bidder:  0.00401  Optimal Revenue:  5.995  payment:  6.813
Batch: 

# Testing

In [11]:
for i in range(200):
    test(50, nbrInit=300, R=300, gamma=0.001, minimum=0, maximum=1)

Total regret:  0.01188 Average regret per bidder:  0.00238  Optimal Revenue:  6.215  payment:  6.832
Total regret:  0.01216 Average regret per bidder:  0.00243  Optimal Revenue:  6.206  payment:  6.831
Total regret:  0.01349 Average regret per bidder:  0.00270  Optimal Revenue:  6.133  payment:  6.792
Total regret:  0.01308 Average regret per bidder:  0.00262  Optimal Revenue:  6.175  payment:  6.825
Total regret:  0.01299 Average regret per bidder:  0.00260  Optimal Revenue:  6.168  payment:  6.814
Total regret:  0.01295 Average regret per bidder:  0.00259  Optimal Revenue:  6.167  payment:  6.812
Total regret:  0.01279 Average regret per bidder:  0.00256  Optimal Revenue:  6.157  payment:  6.798
Total regret:  0.01318 Average regret per bidder:  0.00264  Optimal Revenue:  6.150  payment:  6.801
Total regret:  0.01335 Average regret per bidder:  0.00267  Optimal Revenue:  6.158  payment:  6.814
Total regret:  0.01124 Average regret per bidder:  0.00225  Optimal Revenue:  6.194  paymen

Total regret:  0.01131 Average regret per bidder:  0.00226  Optimal Revenue:  6.242  payment:  6.844
Total regret:  0.01359 Average regret per bidder:  0.00272  Optimal Revenue:  6.100  payment:  6.760
Total regret:  0.01383 Average regret per bidder:  0.00277  Optimal Revenue:  6.094  payment:  6.760
Total regret:  0.01246 Average regret per bidder:  0.00249  Optimal Revenue:  6.160  payment:  6.791
Total regret:  0.01363 Average regret per bidder:  0.00273  Optimal Revenue:  6.175  payment:  6.840
Total regret:  0.01302 Average regret per bidder:  0.00260  Optimal Revenue:  6.140  payment:  6.786
Total regret:  0.01353 Average regret per bidder:  0.00271  Optimal Revenue:  6.205  payment:  6.868
Total regret:  0.01335 Average regret per bidder:  0.00267  Optimal Revenue:  6.143  payment:  6.798
Total regret:  0.01492 Average regret per bidder:  0.00298  Optimal Revenue:  6.023  payment:  6.714
Total regret:  0.01266 Average regret per bidder:  0.00253  Optimal Revenue:  6.222  paymen

Total regret:  0.01395 Average regret per bidder:  0.00279  Optimal Revenue:  6.112  payment:  6.783
Total regret:  0.01434 Average regret per bidder:  0.00287  Optimal Revenue:  6.065  payment:  6.744
Total regret:  0.01166 Average regret per bidder:  0.00233  Optimal Revenue:  6.208  payment:  6.819
Total regret:  0.01212 Average regret per bidder:  0.00242  Optimal Revenue:  6.250  payment:  6.876
Total regret:  0.01293 Average regret per bidder:  0.00259  Optimal Revenue:  6.125  payment:  6.767
Total regret:  0.01286 Average regret per bidder:  0.00257  Optimal Revenue:  6.094  payment:  6.733
Total regret:  0.01185 Average regret per bidder:  0.00237  Optimal Revenue:  6.172  payment:  6.786
Total regret:  0.01194 Average regret per bidder:  0.00239  Optimal Revenue:  6.112  payment:  6.726
Total regret:  0.01383 Average regret per bidder:  0.00277  Optimal Revenue:  5.981  payment:  6.641
Total regret:  0.01280 Average regret per bidder:  0.00256  Optimal Revenue:  6.145  paymen

In [12]:
totalregret = np.mean(np.array(testRegret[-200:]))
revenue     = np.mean(np.array(testPayment[-200:]))
print("Final Result")
print("Total Regret = ", '{0:.5f}'.format(totalregret), "Average regret per bidder: ",'{0:.5f}'.format(totalregret/nAgent), " Optimal Revenue: ",'{0:.3f}'.format(float(np.sqrt(revenue)-np.sqrt(totalregret))**2), " payment: ",'{0:.3f}'.format(revenue))

Final Result
Total Regret =  0.01306 Average regret per bidder:  0.00261  Optimal Revenue:  6.202  payment:  6.784


In [13]:
stdregret = np.std(np.array(testRegret[-200:]))
stdrevenue= np.std(np.array(testPayment[-200:]))
print("std Regret = ", '{0:.5f}'.format(stdregret), "std regret per bidder: ",'{0:.5f}'.format(stdregret/nAgent), " std payment: ",'{0:.3f}'.format(stdrevenue))

std Regret =  0.00105 std regret per bidder:  0.00021  std payment:  0.055


In [14]:
torch.save(mechanism, "510stage2.pth")